In [1]:
pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────────────────────────────────
MASTER_PATH = "/Users/aliabonanno/public_policy_project/master_panel.csv"

# ── LOAD ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(MASTER_PATH, dtype={"FIPS": str})
df["FIPS"] = df["FIPS"].str.zfill(5)

print(f"Loaded: {len(df)} rows, {df['FIPS'].nunique()} counties")

# ── PREP ──────────────────────────────────────────────────────────────────────
# Drop rows missing any key variable
df = df.dropna(subset=["log_res_sales_per_customer", "log_implied_price",
                        "summer_days_95f", "pct_below_150pct_poverty"])

# Demeaned poverty rate (so main heat coefficient is at mean poverty)
mean_pov = df["pct_below_150pct_poverty"].mean()
df["poverty_demeaned"] = df["pct_below_150pct_poverty"] - mean_pov

# Interaction term
df["heat_x_poverty"] = df["summer_days_95f"] * df["poverty_demeaned"]

# Set panel index: entity = FIPS, time = year
df = df.set_index(["FIPS", "year"])

print(f"Analysis sample: {len(df)} obs after dropping missing")
print(f"Mean poverty rate: {mean_pov:.3f}")

# ── HELPER: PRINT RESULTS ─────────────────────────────────────────────────────
def print_results(res, title):
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"N obs:      {int(res.nobs)}")
    print(f"N counties: {res.entity_info['total']}")
    print(f"R-squared (within): {res.rsquared:.4f}")
    print(f"\nKey coefficients:")
    params = res.params
    pvals  = res.pvalues
    ses    = res.std_errors

    for var in ["summer_days_95f", "heat_x_poverty"]:
        if var in params.index:
            stars = ("***" if pvals[var] < 0.01 else
                     "**"  if pvals[var] < 0.05 else
                     "*"   if pvals[var] < 0.10 else "")
            print(f"  {var:30s} {params[var]:8.4f}  "
                  f"(SE: {ses[var]:.4f})  p={pvals[var]:.3f} {stars}")
    print()

# ── SPEC 1: BASELINE — HEAT ONLY ──────────────────────────────────────────────
# log(sales_pc) = β(heat) + α_c + δ_t
mod1 = PanelOLS.from_formula(
    "log_res_sales_per_customer ~ summer_days_95f + EntityEffects + TimeEffects",
    data=df
)
res1 = mod1.fit(cov_type="clustered", cluster_entity=True)
print_results(res1, "SPEC 1: Baseline — Heat only (log sales per customer)")

# ── SPEC 2: POVERTY INTERACTION ───────────────────────────────────────────────
# log(sales_pc) = β(heat) + γ(heat × poverty) + α_c + δ_t
mod2 = PanelOLS.from_formula(
    "log_res_sales_per_customer ~ summer_days_95f + heat_x_poverty "
    "+ EntityEffects + TimeEffects",
    data=df
)
res2 = mod2.fit(cov_type="clustered", cluster_entity=True)
print_results(res2, "SPEC 2: Poverty interaction (log sales per customer)")

# ── SPEC 3: IMPLIED PRICE — BASELINE ─────────────────────────────────────────
mod3 = PanelOLS.from_formula(
    "log_implied_price ~ summer_days_95f + EntityEffects + TimeEffects",
    data=df
)
res3 = mod3.fit(cov_type="clustered", cluster_entity=True)
print_results(res3, "SPEC 3: Baseline — Heat only (log implied price)")

# ── SPEC 4: IMPLIED PRICE — POVERTY INTERACTION ───────────────────────────────
mod4 = PanelOLS.from_formula(
    "log_implied_price ~ summer_days_95f + heat_x_poverty "
    "+ EntityEffects + TimeEffects",
    data=df
)
res4 = mod4.fit(cov_type="clustered", cluster_entity=True)
print_results(res4, "SPEC 4: Poverty interaction (log implied price)")

# ── SPEC 5: HETEROGENEITY BY OWNERSHIP TYPE ───────────────────────────────────
# Add ownership interaction to see if effect differs by utility type
df_own = df.copy()
df_own["is_coop"]      = (df_own["dominant_ownership_type"] == "Cooperative").astype(int)
df_own["is_municipal"] = (df_own["dominant_ownership_type"] == "Municipal").astype(int)
df_own["heat_x_coop"]      = df_own["summer_days_95f"] * df_own["is_coop"]
df_own["heat_x_municipal"]  = df_own["summer_days_95f"] * df_own["is_municipal"]
df_own["heat_pov_x_coop"]   = df_own["heat_x_poverty"]  * df_own["is_coop"]

mod5 = PanelOLS.from_formula(
    "log_res_sales_per_customer ~ summer_days_95f + heat_x_poverty "
    "+ heat_x_coop + heat_x_municipal "
    "+ EntityEffects + TimeEffects",
    data=df_own
)
res5 = mod5.fit(cov_type="clustered", cluster_entity=True)

print(f"\n{'='*60}")
print("SPEC 5: Ownership heterogeneity (log sales per customer)")
print(f"{'='*60}")
print(f"N obs: {int(res5.nobs)}")
for var in ["summer_days_95f", "heat_x_poverty",
            "heat_x_coop", "heat_x_municipal"]:
    if var in res5.params.index:
        p = res5.pvalues[var]
        stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
        print(f"  {var:30s} {res5.params[var]:8.4f}  "
              f"(SE: {res5.std_errors[var]:.4f})  p={p:.3f} {stars}")

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}")
print(f"{'Spec':<8} {'Outcome':<30} {'Heat coef':>10} {'Heat×Pov':>10} {'R2':>8}")
print(f"{'-'*70}")

specs = [
    ("Spec 1", "Log sales/customer",  res1, False),
    ("Spec 2", "Log sales/customer",  res2, True),
    ("Spec 3", "Log implied price",   res3, False),
    ("Spec 4", "Log implied price",   res4, True),
]
for name, outcome, res, has_interaction in specs:
    heat  = res.params.get("summer_days_95f", np.nan)
    inter = res.params.get("heat_x_poverty",  np.nan) if has_interaction else np.nan
    r2    = res.rsquared
    inter_str = f"{inter:10.4f}" if not np.isnan(inter) else f"{'—':>10}"
    print(f"{name:<8} {outcome:<30} {heat:10.4f} {inter_str} {r2:8.4f}")

print(f"\nNote: All specs include county and year fixed effects.")
print(f"Standard errors clustered by county.")
print(f"Heat variable: summer days above 95°F (June–Aug).")
print(f"Poverty variable demeaned at sample mean ({mean_pov:.3f}).")

Loaded: 30928 rows, 3115 counties
Analysis sample: 30685 obs after dropping missing
Mean poverty rate: 0.253

SPEC 1: Baseline — Heat only (log sales per customer)
N obs:      30685
N counties: 3089.0
R-squared (within): 0.0740

Key coefficients:
  summer_days_95f                  0.0013  (SE: 0.0000)  p=0.000 ***


SPEC 2: Poverty interaction (log sales per customer)
N obs:      30685
N counties: 3089.0
R-squared (within): 0.0745

Key coefficients:
  summer_days_95f                  0.0013  (SE: 0.0000)  p=0.000 ***
  heat_x_poverty                  -0.0008  (SE: 0.0004)  p=0.055 *


SPEC 3: Baseline — Heat only (log implied price)
N obs:      30685
N counties: 3089.0
R-squared (within): 0.0013

Key coefficients:
  summer_days_95f                  0.0003  (SE: 0.0001)  p=0.000 ***


SPEC 4: Poverty interaction (log implied price)
N obs:      30685
N counties: 3089.0
R-squared (within): 0.0014

Key coefficients:
  summer_days_95f                  0.0003  (SE: 0.0001)  p=0.000 ***
  hea

In [5]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]
REF_BIN  = "bin6_15to20"
HEAT_BINS = [b for b in BIN_LABELS if b != REF_BIN]
HOT_BINS  = ["bin7_20to25", "bin8_25to30", "bin9_over30"]

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading master panel...")
df = pd.read_csv(f"{OUTPUT_DIR}/master_panel_monthly_fc.csv", dtype={"FIPS": str})
df["FIPS"] = df["FIPS"].str.zfill(5)
print(f"Loaded: {len(df):,} rows, {df['FIPS'].nunique():,} counties")

# ── PREP ──────────────────────────────────────────────────────────────────────
# Drop missing key variables
drop_vars = ["log_res_sales_per_customer", "pct_below_150pct_poverty",
             "prcp_monthly_mm"] + HEAT_BINS
df = df.dropna(subset=drop_vars)
print(f"After dropping missing: {len(df):,} rows")

# Demean key variables
mean_pov = df["pct_below_150pct_poverty"].mean()
mean_fc  = df["fixed_charge_county"].mean()
df["poverty_dm"] = df["pct_below_150pct_poverty"] - mean_pov
df["fc_dm"]      = df["fixed_charge_county"] - mean_fc
df["prcp_sq"]    = df["prcp_monthly_mm"] ** 2

# Month dummies for county-by-month FE via FIPS x year index
for m in range(1, 13):
    df[f"month_{m}"] = (df["month"] == m).astype(int)
month_dummies = " + ".join([f"month_{m}" for m in range(2, 13)])  # month_1 omitted

# Bin x poverty interactions — ALL bins (for coefficient plot)
for b in HEAT_BINS:
    df[f"{b}_x_pov"] = df[b] * df["poverty_dm"]

# Triple interactions
for b in HOT_BINS:
    df[f"{b}_x_pov_x_fc"] = df[b] * df["poverty_dm"] * df["fc_dm"]

# Ownership dummies and triples
df["is_coop"]      = (df["dominant_ownership_type"] == "Cooperative").astype(int)
df["is_municipal"] = (df["dominant_ownership_type"] == "Municipal").astype(int)
for b in HOT_BINS:
    df[f"{b}_x_pov_x_coop"] = df[b] * df["poverty_dm"] * df["is_coop"]
    df[f"{b}_x_pov_x_muni"] = df[b] * df["poverty_dm"] * df["is_municipal"]

# ── INDEX: FIPS x year (cluster by county = cluster_entity) ──────────────────
df = df.set_index(["FIPS", "year"])

print(f"Mean poverty: {mean_pov:.3f} | Mean fixed charge: ${mean_fc:.2f}/month")

# ── FORMULA STRINGS ───────────────────────────────────────────────────────────
bins_str      = " + ".join(HEAT_BINS)
bins_pov_str  = " + ".join([f"{b}_x_pov" for b in HEAT_BINS])  # all bins
hot_pov_str   = " + ".join([f"{b}_x_pov" for b in HOT_BINS])
bins_fc_str   = " + ".join([f"{b}_x_pov_x_fc" for b in HOT_BINS])
bins_coop_str = " + ".join([f"{b}_x_pov_x_coop" for b in HOT_BINS])
bins_muni_str = " + ".join([f"{b}_x_pov_x_muni" for b in HOT_BINS])

# Demographic controls
demo_controls = ("median_hh_income + pct_white + pct_black + "
                 "pct_hispanic + pct_bachelors_plus + unemployment_rate")
controls = f"prcp_monthly_mm + prcp_sq + {month_dummies} + {demo_controls}"

# ── HELPER ────────────────────────────────────────────────────────────────────
def print_results(res, title, key_vars):
    print(f"\n{'='*70}")
    print(f"{title}")
    print(f"{'='*70}")
    print(f"N obs: {int(res.nobs):,}  |  N entities: {int(res.entity_info['total']):,}  |  R²(within): {res.rsquared:.4f}")
    print(f"\nKey coefficients:")
    for var in key_vars:
        if var in res.params.index:
            p = res.pvalues[var]
            stars = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
            print(f"  {var:45s} {res.params[var]:9.4f}  SE={res.std_errors[var]:.4f}  p={p:.3f} {stars}")

# ── SPEC 1: BASELINE ──────────────────────────────────────────────────────────
formula1 = f"log_res_sales_per_customer ~ {bins_str} + {controls} + EntityEffects + TimeEffects"
res1 = PanelOLS.from_formula(formula1, data=df).fit(cov_type="clustered", cluster_entity=True)
print_results(res1, "SPEC 1: Baseline heat response", HEAT_BINS)

# ── SPEC 2: POVERTY INTERACTION (ALL BINS) ────────────────────────────────────
formula2 = (f"log_res_sales_per_customer ~ {bins_str} + {bins_pov_str} "
            f"+ {controls} + EntityEffects + TimeEffects")
res2 = PanelOLS.from_formula(formula2, data=df).fit(cov_type="clustered", cluster_entity=True)
print_results(res2, "SPEC 2: Poverty gap — all bins",
              HEAT_BINS + [f"{b}_x_pov" for b in HEAT_BINS])

# ── SPEC 3: FIXED CHARGE TRIPLE ───────────────────────────────────────────────
df_fc = df.dropna(subset=["fc_dm"])
formula3 = (f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
            f"+ {bins_fc_str} + {controls} + EntityEffects + TimeEffects")
res3 = PanelOLS.from_formula(formula3, data=df_fc).fit(cov_type="clustered", cluster_entity=True)
print_results(res3, "SPEC 3: Fixed charge mechanism",
              [f"{b}_x_pov" for b in HOT_BINS] + [f"{b}_x_pov_x_fc" for b in HOT_BINS])

# ── SPEC 4: IMPLIED PRICE ─────────────────────────────────────────────────────
df_price = df.dropna(subset=["log_implied_price"])
formula4 = (f"log_implied_price ~ {bins_str} + {hot_pov_str} "
            f"+ {controls} + EntityEffects + TimeEffects")
res4 = PanelOLS.from_formula(formula4, data=df_price).fit(cov_type="clustered", cluster_entity=True)
print_results(res4, "SPEC 4: Implied price outcome",
              HOT_BINS + [f"{b}_x_pov" for b in HOT_BINS])

# ── SPEC 5: OWNERSHIP HETEROGENEITY (COOP + MUNICIPAL) ───────────────────────
formula5 = (f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
            f"+ {bins_coop_str} + {bins_muni_str} "
            f"+ {controls} + EntityEffects + TimeEffects")
res5 = PanelOLS.from_formula(formula5, data=df).fit(cov_type="clustered", cluster_entity=True)
print_results(res5, "SPEC 5: Ownership heterogeneity (IOU reference)",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_coop" for b in HOT_BINS] +
              [f"{b}_x_pov_x_muni" for b in HOT_BINS])

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("SUMMARY — KEY INTERACTION COEFFICIENTS")
print(f"{'='*80}")
print(f"{'Variable':<45} {'Spec2':>9} {'Spec3':>9} {'Spec4':>9} {'Spec5':>9}")
print(f"{'-'*80}")

summary_vars = [
    "bin7_20to25_x_pov", "bin8_25to30_x_pov", "bin9_over30_x_pov",
    "bin8_25to30_x_pov_x_fc", "bin9_over30_x_pov_x_fc",
    "bin8_25to30_x_pov_x_coop", "bin9_over30_x_pov_x_coop",
    "bin8_25to30_x_pov_x_muni", "bin9_over30_x_pov_x_muni",
]
for var in summary_vars:
    row = f"  {var:<43}"
    for res in [res2, res3, res4, res5]:
        if var in res.params.index:
            p = res.pvalues[var]
            s = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
            row += f" {res.params[var]:7.4f}{s:<3}"
        else:
            row += f" {'—':>10}"
    print(row)

print(f"\nNote: County FE + year FE + month dummies. SE clustered by county.")
print(f"Controls: precipitation, income, race, education, unemployment.")
print(f"Reference: bin 15-20C, IOU ownership. Poverty and FC demeaned.")

# ── SAVE COEFFICIENTS FOR COEFFICIENT PLOT ────────────────────────────────────
# Extract all bin x poverty coefficients from Spec 2 for plotting
coef_rows = []
bin_names = {
    "bin1_under_neg5_x_pov": "< −5°C",
    "bin2_neg5to0_x_pov":    "−5–0°C",
    "bin3_0to5_x_pov":       "0–5°C",
    "bin4_5to10_x_pov":      "5–10°C",
    "bin5_10to15_x_pov":     "10–15°C",
    "bin7_20to25_x_pov":     "20–25°C",
    "bin8_25to30_x_pov":     "25–30°C",
    "bin9_over30_x_pov":     "> 30°C",
}
# Reference bin
coef_rows.append({"bin": "15–20°C", "coef": 0.0, "se": 0.0, "is_ref": True})
for var, label in bin_names.items():
    if var in res2.params.index:
        coef_rows.append({
            "bin": label,
            "coef": res2.params[var],
            "se": res2.std_errors[var],
            "is_ref": False
        })

coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(f"{OUTPUT_DIR}/coef_plot_data.csv", index=False)
print(f"\nSaved coef_plot_data.csv for coefficient plot")

Loading master panel...
Loaded: 335,878 rows, 2,814 counties
After dropping missing: 333,163 rows
Mean poverty: 0.251 | Mean fixed charge: $11.87/month

SPEC 1: Baseline heat response
N obs: 333,127  |  N entities: 2,793  |  R²(within): 0.7048

Key coefficients:
  bin1_under_neg5                                  0.0134  SE=0.0004  p=0.000 ***
  bin2_neg5to0                                     0.0079  SE=0.0003  p=0.000 ***
  bin3_0to5                                        0.0102  SE=0.0002  p=0.000 ***
  bin4_5to10                                       0.0065  SE=0.0002  p=0.000 ***
  bin5_10to15                                     -0.0005  SE=0.0003  p=0.041 **
  bin7_20to25                                      0.0060  SE=0.0002  p=0.000 ***
  bin8_25to30                                      0.0112  SE=0.0003  p=0.000 ***
  bin9_over30                                      0.0199  SE=0.0006  p=0.000 ***

SPEC 2: Poverty gap — all bins
N obs: 333,127  |  N entities: 2,793  |  R²(within

In [16]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]
REF_BIN   = "bin6_15to20"
HEAT_BINS = [b for b in BIN_LABELS if b != REF_BIN]
HOT_BINS  = ["bin7_20to25", "bin8_25to30", "bin9_over30"]

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading master panel...")
df = pd.read_csv(f"{OUTPUT_DIR}/master_panel_monthly_fc.csv", dtype={"FIPS": str})
df["FIPS"] = df["FIPS"].str.zfill(5)
print(f"Loaded: {len(df):,} rows, {df['FIPS'].nunique():,} counties")

# ── PREP ──────────────────────────────────────────────────────────────────────
drop_vars = ["log_res_sales_per_customer", "pct_below_150pct_poverty",
             "prcp_monthly_mm"] + HEAT_BINS
df = df.dropna(subset=drop_vars)
print(f"After dropping missing: {len(df):,} rows")

# Demean
mean_pov = df["pct_below_150pct_poverty"].mean()
mean_fc  = df["fixed_charge_county"].mean()
df["poverty_dm"] = df["pct_below_150pct_poverty"] - mean_pov
df["fc_dm"]      = df["fixed_charge_county"] - mean_fc
df["prcp_sq"]    = df["prcp_monthly_mm"] ** 2

# Rural dummy RUCC >= 7
df["rural"] = (df["rucc_score"] >= 7).astype(int)

# Month dummies
for m in range(1, 13):
    df[f"month_{m}"] = (df["month"] == m).astype(int)
month_dummies = " + ".join([f"month_{m}" for m in range(2, 13)])

# COVID indicator
df["covid"] = (df["year"] == 2020).astype(int)

# Bin x poverty — all bins
for b in HEAT_BINS:
    df[f"{b}_x_pov"] = df[b] * df["poverty_dm"]

# Triple interactions — ownership, fixed charge, rural, COVID
for b in HOT_BINS:
    df[f"{b}_x_pov_x_fc"]    = df[b] * df["poverty_dm"] * df["fc_dm"]
    df[f"{b}_x_pov_x_coop"]  = df[b] * df["poverty_dm"] * df["is_coop"]
    df[f"{b}_x_pov_x_muni"]  = df[b] * df["poverty_dm"] * df["is_municipal"]
    df[f"{b}_x_pov_x_rural"] = df[b] * df["poverty_dm"] * df["rural"]
    df[f"{b}_x_covid"]       = df[b] * df["covid"]

# Average price level mechanism (Spec 8)
# County-mean log implied price as time-invariant structural price measure
county_price = (df.reset_index()
                .groupby("FIPS")["log_implied_price"]
                .mean().reset_index()
                .rename(columns={"log_implied_price": "mean_log_price"}))
df = df.reset_index().merge(county_price, on="FIPS", how="left").set_index(["FIPS", "year"])
mean_price = df["mean_log_price"].mean()
df["price_dm"] = df["mean_log_price"] - mean_price
for b in HOT_BINS:
    df[f"{b}_x_pov_x_price"] = df[b] * df["poverty_dm"] * df["price_dm"]

print(f"Mean poverty: {mean_pov:.3f} | Mean fixed charge: ${mean_fc:.2f}/month")
print(f"Mean county avg log price: {mean_price:.3f} (${np.exp(mean_price):.2f}/MWh)")

# ── FORMULA STRINGS ───────────────────────────────────────────────────────────
bins_str       = " + ".join(HEAT_BINS)
bins_pov_str   = " + ".join([f"{b}_x_pov" for b in HEAT_BINS])
hot_pov_str    = " + ".join([f"{b}_x_pov" for b in HOT_BINS])
bins_fc_str    = " + ".join([f"{b}_x_pov_x_fc" for b in HOT_BINS])
bins_coop_str  = " + ".join([f"{b}_x_pov_x_coop" for b in HOT_BINS])
bins_muni_str  = " + ".join([f"{b}_x_pov_x_muni" for b in HOT_BINS])
bins_rural_str = " + ".join([f"{b}_x_pov_x_rural" for b in HOT_BINS])
bins_covid_str = " + ".join([f"{b}_x_covid" for b in HOT_BINS])
bins_price_str = " + ".join([f"{b}_x_pov_x_price" for b in HOT_BINS])

# Base controls
base_controls = (f"prcp_monthly_mm + prcp_sq + {month_dummies} + "
                 f"median_hh_income + pct_white + pct_black + "
                 f"pct_hispanic + pct_bachelors_plus + unemployment_rate + "
                 f"pct_electric_heat")

# ── HELPER ────────────────────────────────────────────────────────────────────
def print_results(res, title, key_vars):
    print(f"\n{'='*70}")
    print(f"{title}")
    print(f"{'='*70}")
    print(f"N obs: {int(res.nobs):,}  |  "
          f"N entities: {int(res.entity_info['total']):,}  |  "
          f"R²(within): {res.rsquared:.4f}")
    print(f"\nKey coefficients:")
    for var in key_vars:
        if var in res.params.index:
            p = res.pvalues[var]
            stars = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
            print(f"  {var:48s} {res.params[var]:9.4f}  "
                  f"SE={res.std_errors[var]:.4f}  p={p:.3f} {stars}")

def fit(formula, data=None):
    d = df if data is None else data
    return PanelOLS.from_formula(
        formula, data=d).fit(cov_type="clustered", cluster_entity=True)

# ── SPEC 1: BASELINE ──────────────────────────────────────────────────────────
res1 = fit(f"log_res_sales_per_customer ~ {bins_str} + {base_controls} "
           f"+ EntityEffects + TimeEffects")
print_results(res1, "SPEC 1: Baseline heat response", HEAT_BINS)

# ── SPEC 2: POVERTY GAP (ALL BINS) ───────────────────────────────────────────
res2 = fit(f"log_res_sales_per_customer ~ {bins_str} + {bins_pov_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects")
print_results(res2, "SPEC 2: Poverty gap — all bins",
              HEAT_BINS + [f"{b}_x_pov" for b in HEAT_BINS])

# ── SPEC 3: FIXED CHARGE ──────────────────────────────────────────────────────
df_fc = df.dropna(subset=["fc_dm"])
res3 = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
           f"+ {bins_fc_str} + {base_controls} + EntityEffects + TimeEffects",
           data=df_fc)
print_results(res3, "SPEC 3: Fixed charge mechanism",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_fc" for b in HOT_BINS])

# ── SPEC 4: IMPLIED PRICE OUTCOME ─────────────────────────────────────────────
df_price_out = df.dropna(subset=["log_implied_price"])
res4 = fit(f"log_implied_price ~ {bins_str} + {hot_pov_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects",
           data=df_price_out)
print_results(res4, "SPEC 4: Implied price outcome",
              HOT_BINS + [f"{b}_x_pov" for b in HOT_BINS])

# ── SPEC 5: OWNERSHIP (COOP + PUBLIC) ────────────────────────────────────────
res5 = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
           f"+ {bins_coop_str} + {bins_muni_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects")
print_results(res5, "SPEC 5: Ownership heterogeneity (IOU reference)",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_coop" for b in HOT_BINS] +
              [f"{b}_x_pov_x_muni" for b in HOT_BINS])

# ── SPEC 6: OWNERSHIP + RURALITY ─────────────────────────────────────────────
res6 = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
           f"+ {bins_coop_str} + {bins_muni_str} + {bins_rural_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects")
print_results(res6, "SPEC 6: Ownership + rurality control",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_coop" for b in HOT_BINS] +
              [f"{b}_x_pov_x_muni" for b in HOT_BINS] +
              [f"{b}_x_pov_x_rural" for b in HOT_BINS])

# ── SPEC 7a: EXCLUDE 2020 ────────────────────────────────────────────────────
df_nocovid = df[df.index.get_level_values("year") != 2020]
res7a = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
            f"+ {bins_coop_str} + {bins_muni_str} "
            f"+ {base_controls} + EntityEffects + TimeEffects",
            data=df_nocovid)
print_results(res7a, "SPEC 7a: Ownership — excluding 2020 (COVID)",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_coop" for b in HOT_BINS] +
              [f"{b}_x_pov_x_muni" for b in HOT_BINS])

# ── SPEC 7b: COVID INTERACTION ───────────────────────────────────────────────
res7b = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
            f"+ {bins_coop_str} + {bins_muni_str} + {bins_covid_str} "
            f"+ {base_controls} + EntityEffects + TimeEffects")
print_results(res7b, "SPEC 7b: Ownership + COVID interaction",
              [f"{b}_x_pov_x_coop" for b in HOT_BINS] +
              [f"{b}_x_covid" for b in HOT_BINS])

# ── SPEC 8: AVERAGE PRICE LEVEL MECHANISM ────────────────────────────────────
# Tests whether lower absolute price in public utility counties explains
# the attenuation of the poverty gap.
# Hypothesis: lambda_b^price > 0 — higher average county price amplifies
# the poverty gap because the absolute budget constraint is tighter.
df_p = df.dropna(subset=["price_dm"])
res8 = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
           f"+ {bins_price_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects",
           data=df_p)
print_results(res8, "SPEC 8: Average price level mechanism",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_price" for b in HOT_BINS])

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("SUMMARY — KEY COEFFICIENTS ACROSS SPECS")
print(f"{'='*90}")
specs  = [res2, res3, res5, res6, res7a, res8]
labels = ["Spec2", "Spec3", "Spec5", "Spec6", "Spec7a", "Spec8"]
print(f"{'Variable':<50} " + " ".join(f"{l:>9}" for l in labels))
print("-"*100)

summary_vars = [
    "bin7_20to25_x_pov", "bin8_25to30_x_pov", "bin9_over30_x_pov",
    "bin8_25to30_x_pov_x_fc",    "bin9_over30_x_pov_x_fc",
    "bin8_25to30_x_pov_x_coop",  "bin9_over30_x_pov_x_coop",
    "bin8_25to30_x_pov_x_muni",  "bin9_over30_x_pov_x_muni",
    "bin8_25to30_x_pov_x_rural", "bin9_over30_x_pov_x_rural",
    "bin8_25to30_x_pov_x_price", "bin9_over30_x_pov_x_price",
]
for var in summary_vars:
    row = f"  {var:<48}"
    for res in specs:
        if var in res.params.index:
            p = res.pvalues[var]
            s = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
            row += f" {res.params[var]:7.4f}{s:<2}"
        else:
            row += f" {'—':>9}"
    print(row)

print(f"\nNote: County FE + year FE + month dummies. SE clustered by county.")
print(f"Controls: precipitation, income, race, education, unemployment, electric heat share.")
print(f"Spec 8: county mean log implied price as structural price measure.")
print(f"Reference: bin 15-20C, IOU ownership. Poverty and price demeaned.")

# ── COEFFICIENT PLOT DATA ─────────────────────────────────────────────────────
coef_rows = []
bin_names = {
    "bin1_under_neg5_x_pov": "< −5°C",
    "bin2_neg5to0_x_pov":    "−5–0°C",
    "bin3_0to5_x_pov":       "0–5°C",
    "bin4_5to10_x_pov":      "5–10°C",
    "bin5_10to15_x_pov":     "10–15°C",
    "bin7_20to25_x_pov":     "20–25°C",
    "bin8_25to30_x_pov":     "25–30°C",
    "bin9_over30_x_pov":     "> 30°C",
}
coef_rows.append({"bin": "15–20°C", "coef": 0.0, "se": 0.0, "is_ref": True})
for var, label in bin_names.items():
    if var in res2.params.index:
        coef_rows.append({
            "bin": label,
            "coef": res2.params[var],
            "se": res2.std_errors[var],
            "is_ref": False
        })
pd.DataFrame(coef_rows).to_csv(f"{OUTPUT_DIR}/coef_plot_data.csv", index=False)
print(f"\nSaved coef_plot_data.csv")

Loading master panel...
Loaded: 335,878 rows, 2,814 counties
After dropping missing: 333,163 rows
Mean poverty: 0.251 | Mean fixed charge: $11.87/month
Mean county avg log price: 4.836 ($125.92/MWh)

SPEC 1: Baseline heat response
N obs: 333,127  |  N entities: 2,793  |  R²(within): 0.7048

Key coefficients:
  bin1_under_neg5                                     0.0134  SE=0.0004  p=0.000 ***
  bin2_neg5to0                                        0.0079  SE=0.0003  p=0.000 ***
  bin3_0to5                                           0.0102  SE=0.0002  p=0.000 ***
  bin4_5to10                                          0.0065  SE=0.0002  p=0.000 ***
  bin5_10to15                                        -0.0005  SE=0.0003  p=0.040 **
  bin7_20to25                                         0.0060  SE=0.0002  p=0.000 ***
  bin8_25to30                                         0.0112  SE=0.0003  p=0.000 ***
  bin9_over30                                         0.0199  SE=0.0006  p=0.000 ***

SPEC 2: Po

In [3]:
import pandas as pd
import numpy as np

OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

master = pd.read_csv(f"{OUTPUT_DIR}/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})
county_tou = pd.read_csv(f"{OUTPUT_DIR}/county_tou_prevalence.csv",
                         dtype={"FIPS": str})
county_tou["FIPS"] = county_tou["FIPS"].str.zfill(5)

master = master.merge(county_tou[["FIPS","tou_prevalence"]],
                      on="FIPS", how="left")

# Demean TOU prevalence
mean_tou = master["tou_prevalence"].mean()
master["tou_demeaned"] = master["tou_prevalence"] - mean_tou

# Triple interactions with TOU
HOT_BINS = ["bin7_20to25","bin8_25to30","bin9_over30"]
for b in HOT_BINS:
    master[f"{b}_x_pov_x_tou"] = (master[b] * 
                                    master["poverty_demeaned"] * 
                                    master["tou_demeaned"])

master.to_csv(f"{OUTPUT_DIR}/master_panel_monthly_fc.csv", index=False)
print(f"Mean TOU prevalence: {mean_tou:.3f}")
print(f"Missing TOU: {master['tou_prevalence'].isna().sum()}")
print("Saved.")

Mean TOU prevalence: 0.834
Missing TOU: 0
Saved.


In [5]:
bins_tou_str = " + ".join([f"{b}_x_pov_x_tou" for b in HOT_BINS])

res8 = fit(f"log_res_sales_per_customer ~ {bins_str} + {hot_pov_str} "
           f"+ {bins_tou_str} "
           f"+ {base_controls} + EntityEffects + TimeEffects")
print_results(res8, "SPEC 8: TOU prevalence mechanism",
              [f"{b}_x_pov" for b in HOT_BINS] +
              [f"{b}_x_pov_x_tou" for b in HOT_BINS])

NameError: name 'fit' is not defined

In [11]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

print("Mean implied price by ownership type ($/MWh = cents/kWh * 10):")
result = (master.groupby("dominant_ownership_type")["log_implied_price"]
          .agg(["mean","count"])
          .assign(
              implied_price_mwh=lambda x: np.exp(x["mean"]),
              implied_price_kwh_cents=lambda x: np.exp(x["mean"]) / 10
          )
          .round(3))
print(result)

print("\nAnnual bill estimate (assuming 10.8 MWh/year):")
for idx, row in result.iterrows():
    annual = row["implied_price_mwh"] * 10.8
    print(f"  {idx}: ${annual:.0f}/year")

Mean implied price by ownership type ($/MWh = cents/kWh * 10):
                          mean   count  implied_price_mwh  \
dominant_ownership_type                                     
Cooperative              4.848   19332            127.461   
IOU                      4.849  302567            127.592   
Municipal                4.692   13968            109.106   

                         implied_price_kwh_cents  
dominant_ownership_type                           
Cooperative                               12.746  
IOU                                       12.759  
Municipal                                 10.911  

Annual bill estimate (assuming 10.8 MWh/year):
  Cooperative: $1377/year
  IOU: $1378/year
  Municipal: $1178/year


In [1]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

vars_of_interest = [
    "res_sales_per_customer",
    "implied_price_per_mwh", 
    "fixed_charge_county",
    "pct_below_150pct_poverty",
    "median_hh_income",
    "pct_electric_heat",
    "unemployment_rate",
    "bin7_20to25",
    "bin8_25to30", 
    "bin9_over30",
    "prcp_monthly_mm"
]

print(master[vars_of_interest].describe().round(3).to_string())

# By ownership type
for var in ["implied_price_per_mwh", "fixed_charge_county", "pct_below_150pct_poverty"]:
    print(f"\n{var} by ownership:")
    print(master.groupby("dominant_ownership_type")[var].agg(["mean","std","count"]).round(3))

/Users/aliabonanno/Downloads/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


       res_sales_per_customer  implied_price_per_mwh  fixed_charge_county  pct_below_150pct_poverty  median_hh_income  pct_electric_heat  unemployment_rate  bin7_20to25  bin8_25to30  bin9_over30  prcp_monthly_mm
count              335878.000             335867.000           335878.000                335578.000        335542.000         335590.000         335578.000   333466.000   333466.000   333466.000       333466.000
mean                    0.944                131.183               11.864                     0.251         56321.044              0.049              5.732        5.738        3.720        0.206           87.130
std                     0.307                 41.616                4.487                     0.086         16656.690              0.066              2.913        7.292        7.522        1.530           63.206
min                     0.000                 12.932                0.000                     0.012         17109.000              0.000              0.

In [3]:
# Regression sample only
reg_sample = master.dropna(subset=vars_of_interest).copy()
print(f"Regression sample N: {len(reg_sample)}")
print(f"Unique counties: {reg_sample['FIPS'].nunique()}")
print(reg_sample[vars_of_interest].describe().round(3).to_string())

Regression sample N: 333119
Unique counties: 2793
       res_sales_per_customer  implied_price_per_mwh  fixed_charge_county  pct_below_150pct_poverty  median_hh_income  pct_electric_heat  unemployment_rate  bin7_20to25  bin8_25to30  bin9_over30  prcp_monthly_mm
count              333119.000             333119.000           333119.000                333119.000        333119.000         333119.000         333119.000   333119.000   333119.000   333119.000       333119.000
mean                    0.947                129.625               11.869                     0.251         56194.416              0.049              5.720        5.739        3.722        0.205           87.121
std                     0.306                 35.526                4.488                     0.086         16603.096              0.065              2.885        7.292        7.525        1.528           63.204
min                     0.001                 12.932                0.000                     0.026   

In [5]:
print(state_stats[state_stats["state"] == "AZ"][["state","coop_share"]].to_string())

NameError: name 'state_stats' is not defined

In [6]:

import pandas as pd
import numpy as np

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

print("="*60)
print("1. COUNTY COUNTS")
print("="*60)
total_counties = master["FIPS"].nunique()
print(f"Total unique counties in master panel: {total_counties}")
by_type = master.groupby("dominant_ownership_type")["FIPS"].nunique()
print(f"By ownership:\n{by_type}")
print(f"Sum: {by_type.sum()}")

reg_vars = ["res_sales_per_customer","implied_price_per_mwh","fixed_charge_county",
            "pct_below_150pct_poverty","median_hh_income","pct_electric_heat",
            "unemployment_rate","bin7_20to25","bin8_25to30","bin9_over30","prcp_monthly_mm"]
reg = master.dropna(subset=reg_vars)
print(f"\nRegression sample N obs: {len(reg)}")
print(f"Regression sample counties: {reg['FIPS'].nunique()}")
reg_by_type = reg.groupby("dominant_ownership_type")["FIPS"].nunique()
print(f"Regression counties by type:\n{reg_by_type}")

print("\n" + "="*60)
print("2. FIXED CHARGES BY OWNERSHIP")
print("="*60)
print("Full sample:")
print(master.groupby("dominant_ownership_type")["fixed_charge_county"].mean().round(3))
print("Regression sample:")
print(reg.groupby("dominant_ownership_type")["fixed_charge_county"].mean().round(3))

print("\n" + "="*60)
print("3. FOREGONE KWH CALCULATION")
print("="*60)
lambda8 = -0.0248
lambda9 = -0.0378
coop_summer = master[(master["dominant_ownership_type"]=="Cooperative") &
                      (master["month"].isin([6,7,8]))]
bin8_mean = coop_summer["bin8_25to30"].mean()
bin9_mean = coop_summer["bin9_over30"].mean()
sales_kwh = coop_summer["res_sales_per_customer"].mean() * 1000
p75 = master["pct_below_150pct_poverty"].quantile(0.75)
mean_pov = master["pct_below_150pct_poverty"].mean()
contrast = p75 - mean_pov
log_effect = (lambda8 * bin8_mean * 3 * contrast) + (lambda9 * bin9_mean * 3 * contrast)
foregone_kwh = sales_kwh * 3 * (1 - np.exp(log_effect))
coop_price = master[master["dominant_ownership_type"]=="Cooperative"]["implied_price_per_mwh"].mean()
foregone_dollars = foregone_kwh * (coop_price / 1000)
print(f"P75: {p75:.4f}, Mean: {mean_pov:.4f}, Contrast: {contrast:.4f}")
print(f"Bin8 coop summer: {bin8_mean:.4f}, Bin9: {bin9_mean:.4f}")
print(f"Mean sales coop summer: {sales_kwh:.2f} kWh/month")
print(f"Foregone kWh: {foregone_kwh:.1f}")
print(f"Foregone hours (÷3.5): {foregone_kwh/3.5:.1f}")
print(f"Coop price: ${coop_price:.2f}/MWh")
print(f"Foregone $: ${foregone_dollars:.2f}")

print("\n" + "="*60)
print("4. AGGREGATE SHORTFALL")
print("="*60)
eligible = 9_836_114 * mean_pov
aggregate = eligible * foregone_dollars
print(f"Eligible coop customers: {eligible:,.0f}")
print(f"Aggregate shortfall: ${aggregate/1e6:.1f}M")

print("\n" + "="*60)
print("5. OBS COUNT CHECK")
print("="*60)
print(f"Full panel rows: {len(master)}")
print(f"Regression sample rows: {len(reg)}")
summer_coop = master[(master["dominant_ownership_type"]=="Cooperative") &
                      (master["month"].isin([6,7,8]))]
print(f"Summer coop obs: {len(summer_coop)}")


1. COUNTY COUNTS
Total unique counties in master panel: 2814
By ownership:
dominant_ownership_type
Cooperative     172
IOU            2539
Municipal       118
Name: FIPS, dtype: int64
Sum: 2829

Regression sample N obs: 333119
Regression sample counties: 2793
Regression counties by type:
dominant_ownership_type
Cooperative     160
IOU            2531
Municipal       117
Name: FIPS, dtype: int64

2. FIXED CHARGES BY OWNERSHIP
Full sample:
dominant_ownership_type
Cooperative    17.068
IOU            11.308
Municipal      16.719
Name: fixed_charge_county, dtype: float64
Regression sample:
dominant_ownership_type
Cooperative    17.604
IOU            11.302
Municipal      16.765
Name: fixed_charge_county, dtype: float64

3. FOREGONE KWH CALCULATION
P75: 0.3031, Mean: 0.2507, Contrast: 0.0525
Bin8 coop summer: 11.9103, Bin9: 1.0500
Mean sales coop summer: 1142.86 kWh/month
Foregone kWh: 176.2
Foregone hours (÷3.5): 50.3
Coop price: $137.92/MWh
Foregone $: $24.30

4. AGGREGATE SHORTFALL
Eligi